In [ ]:
import os

# 기본 경로
base_path = r"E:\projects\SpaceUp\Track2\sentinel-2"

# 찾고자 하는 폴더 이름 목록
target_folders = ["Dammam", "Dhahran", "Madinah", "Riyadh"]

# 실제 존재하는 폴더만 추출
found_folders = []

for folder in target_folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        found_folders.append(folder_path)

# 결과 출력
if found_folders:
    print("다음 폴더를 찾았습니다:")
    for path in found_folders:
        print(path)
else:
    print("지정한 폴더를 찾을 수 없습니다.")


In [ ]:
import os

# 타겟 폴더들
base_folders = [
    r"E:\projects\SpaceUp\Track2\sentinel-2\Dammam",
    r"E:\projects\SpaceUp\Track2\sentinel-2\Dhahran",
    r"E:\projects\SpaceUp\Track2\sentinel-2\Madinah",
    r"E:\projects\SpaceUp\Track2\sentinel-2\Riyadh"
]

# 각 폴더에서 .SAFE 디렉토리 찾기
for base in base_folders:
    print(f"\n[{os.path.basename(base)}] 폴더 내 .SAFE 디렉터리:")
    for item in os.listdir(base):
        item_path = os.path.join(base, item)
        if os.path.isdir(item_path) and item.endswith(".SAFE"):
            print(f"  - {item_path}")


In [ ]:
import os

# 도시별 기본 경로
base_folders = [
    r"E:\projects\SpaceUp\Track2\sentinel-2\Dammam",
    r"E:\projects\SpaceUp\Track2\sentinel-2\Dhahran",
    r"E:\projects\SpaceUp\Track2\sentinel-2\Madinah",
    r"E:\projects\SpaceUp\Track2\sentinel-2\Riyadh"
]

# 생성할 하위 폴더 이름
subfolders = ["RGB", "NDVI"]

# 폴더 생성
for city_path in base_folders:
    for sub in subfolders:
        new_folder_path = os.path.join(city_path, sub)
        os.makedirs(new_folder_path, exist_ok=True)
        print(f"생성됨: {new_folder_path}")


In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from datetime import datetime

# 도시 폴더 경로 설정
base_dir = r"E:\projects\SpaceUp\Track2\sentinel-2"
cities = ["Dammam", "Dhahran", "Madinah", "Riyadh"]
target_crs = "EPSG:4326"

# 필요한 밴드
bands = {
    "B02": None,  # Blue
    "B03": None,  # Green
    "B04": None,  # Red
    "B08": None   # NIR
}

def find_band_paths(safe_dir):
    granule_dir = os.path.join(safe_dir, "GRANULE")
    if not os.path.exists(granule_dir):
        return None

    l2a_dir = [d for d in os.listdir(granule_dir) if d.startswith("L2A")]
    if not l2a_dir:
        return None

    r10m_path = os.path.join(granule_dir, l2a_dir[0], "IMG_DATA", "R10m")
    if not os.path.exists(r10m_path):
        return None

    files = os.listdir(r10m_path)
    result = {}

    for f in files:
        for band in bands:
            if f.endswith(f"{band}_10m.jp2"):
                result[band] = os.path.join(r10m_path, f)

    return result if len(result) == 4 else None

def load_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
    return arr, profile

def reproject_band(array, profile, dst_crs, fixed_transform=None, fixed_width=None, fixed_height=None):
    src_crs = profile["crs"]
    height, width = array.shape

    if fixed_transform is None:
        # 기준 transform 계산
        with rasterio.io.MemoryFile() as memfile:
            with memfile.open(
                driver="GTiff",
                height=height,
                width=width,
                count=1,
                dtype="float32",
                crs=src_crs,
                transform=profile["transform"]
            ) as tmp_dst:
                tmp_dst.write(array, 1)
                bounds = tmp_dst.bounds

        transform, width, height = calculate_default_transform(
            src_crs, dst_crs, width, height,
            left=bounds.left, bottom=bounds.bottom,
            right=bounds.right, top=bounds.top
        )
    else:
        # 고정된 transform과 크기를 사용
        transform = fixed_transform
        width = fixed_width
        height = fixed_height

    dst_array = np.empty((height, width), dtype="float32")
    reproject(
        source=array,
        destination=dst_array,
        src_transform=profile["transform"],
        src_crs=src_crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

    profile.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "dtype": "float32"
    })

    return dst_array, profile, transform, width, height

def save_tiff(path, array, profile, count=1):
    profile.update(driver="GTiff", count=count)
    with rasterio.open(path, "w", **profile) as dst:
        if count == 1:
            dst.write(array, 1)
        else:
            for i in range(count):
                dst.write(array[i], i + 1)

# 메인 실행
for city in cities:
    city_path = os.path.join(base_dir, city)
    rgb_dir = os.path.join(city_path, "RGB")
    ndvi_dir = os.path.join(city_path, "NDVI")
    os.makedirs(rgb_dir, exist_ok=True)
    os.makedirs(ndvi_dir, exist_ok=True)

    for item in os.listdir(city_path):
        if not item.endswith(".SAFE"):
            continue

        safe_path = os.path.join(city_path, item)
        band_paths = find_band_paths(safe_path)
        if not band_paths:
            print(f"[{city}] {item} - 필요한 밴드 누락됨")
            continue

        # 날짜 추출
        try:
            date_str = item.split("_")[2][:8]
            date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y%m%d")
        except:
            date_fmt = "unknown"

        # 밴드 로드
        b02, profile = load_band(band_paths["B02"])
        b03, _ = load_band(band_paths["B03"])
        b04, _ = load_band(band_paths["B04"])
        b08, _ = load_band(band_paths["B08"])

        # RGB 스택 및 NDVI 계산
        rgb = np.stack([b04, b03, b02])
        ndvi = (b08 - b04) / (b08 + b04 + 1e-6)

        # RGB 재투영 (첫 밴드 기준 고정)
        rgb_reproj = []
        fixed_transform = None
        fixed_width = None
        fixed_height = None
        reproj_profile = None

        for i in range(3):
            if i == 0:
                r_band, reproj_profile, fixed_transform, fixed_width, fixed_height = reproject_band(
                    rgb[i], profile, target_crs
                )
            else:
                r_band, _, _, _, _ = reproject_band(
                    rgb[i], profile, target_crs,
                    fixed_transform=fixed_transform,
                    fixed_width=fixed_width,
                    fixed_height=fixed_height
                )
            rgb_reproj.append(r_band)
        rgb_reproj = np.stack(rgb_reproj)

        # NDVI 재투영 (RGB와 동일한 기준 사용)
        ndvi_reproj, ndvi_profile, _, _, _ = reproject_band(
            ndvi, profile, target_crs,
            fixed_transform=fixed_transform,
            fixed_width=fixed_width,
            fixed_height=fixed_height
        )

        # 저장 경로
        rgb_name = f"{city}_{date_fmt}_RGB.tiff"
        ndvi_name = f"{city}_{date_fmt}_NDVI.tiff"

        rgb_out = os.path.join(rgb_dir, rgb_name)
        ndvi_out = os.path.join(ndvi_dir, ndvi_name)

        save_tiff(rgb_out, rgb_reproj, reproj_profile, count=3)
        save_tiff(ndvi_out, ndvi_reproj, ndvi_profile, count=1)

        print(f"[{city}] 저장 완료: {rgb_name}, {ndvi_name}")


In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from datetime import datetime

# 도시 폴더 경로 설정
base_dir = r"E:\projects\SpaceUp\Track2\sentinel-2"
cities = ["Dammam", "Dhahran", "Madinah", "Riyadh"]
target_crs = "EPSG:4326"

# 필요한 밴드
bands = {
    "B02": None,  # Blue
    "B03": None,  # Green
    "B04": None,  # Red
    "B08": None   # NIR
}

def find_band_paths(safe_dir):
    granule_dir = os.path.join(safe_dir, "GRANULE")
    if not os.path.exists(granule_dir):
        return None

    l2a_dir = [d for d in os.listdir(granule_dir) if d.startswith("L2A")]
    if not l2a_dir:
        return None

    r10m_path = os.path.join(granule_dir, l2a_dir[0], "IMG_DATA", "R10m")
    if not os.path.exists(r10m_path):
        return None

    files = os.listdir(r10m_path)
    result = {}

    for f in files:
        for band in bands:
            if f.endswith(f"{band}_10m.jp2"):
                result[band] = os.path.join(r10m_path, f)

    return result if len(result) == 4 else None

def load_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
    return arr, profile

def reproject_band(array, profile, dst_crs, fixed_transform=None, fixed_width=None, fixed_height=None):
    src_crs = profile["crs"]
    height, width = array.shape

    if fixed_transform is None:
        with rasterio.io.MemoryFile() as memfile:
            with memfile.open(
                driver="GTiff",
                height=height,
                width=width,
                count=1,
                dtype="float32",
                crs=src_crs,
                transform=profile["transform"]
            ) as tmp_dst:
                tmp_dst.write(array, 1)
                bounds = tmp_dst.bounds

        transform, width, height = calculate_default_transform(
            src_crs, dst_crs, width, height,
            left=bounds.left, bottom=bounds.bottom,
            right=bounds.right, top=bounds.top
        )
    else:
        transform = fixed_transform
        width = fixed_width
        height = fixed_height

    dst_array = np.empty((height, width), dtype="float32")
    reproject(
        source=array,
        destination=dst_array,
        src_transform=profile["transform"],
        src_crs=src_crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

    profile.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "dtype": "float32"
    })

    return dst_array, profile, transform, width, height

def save_tiff(path, array, profile, count=1):
    profile.update(driver="GTiff", count=count)
    with rasterio.open(path, "w", **profile) as dst:
        if count == 1:
            dst.write(array, 1)
        else:
            for i in range(count):
                dst.write(array[i], i + 1)

# 메인 실행
for city in cities:
    city_path = os.path.join(base_dir, city)
    rgb_dir = os.path.join(city_path, "RGB")
    ndvi_dir = os.path.join(city_path, "NDVI")
    os.makedirs(rgb_dir, exist_ok=True)
    os.makedirs(ndvi_dir, exist_ok=True)

    for item in os.listdir(city_path):
        if not item.endswith(".SAFE"):
            continue

        safe_path = os.path.join(city_path, item)
        band_paths = find_band_paths(safe_path)
        if not band_paths:
            print(f"[{city}] {item} - 필요한 밴드 누락됨")
            continue

        try:
            date_str = item.split("_")[2][:8]
            date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y%m%d")
        except:
            date_fmt = "unknown"

        # 밴드 로드
        b02, profile = load_band(band_paths["B02"])
        b03, _ = load_band(band_paths["B03"])
        b04, _ = load_band(band_paths["B04"])
        b08, _ = load_band(band_paths["B08"])

        # RGB 스택 및 NDVI 계산
        rgb = np.stack([b04, b03, b02])
        ndvi = (b08 - b04) / (b08 + b04 + 1e-6)

        # RGB 재투영
        rgb_reproj = []
        fixed_transform = None
        fixed_width = None
        fixed_height = None
        reproj_profile = None

        for i in range(3):
            if i == 0:
                r_band, reproj_profile, fixed_transform, fixed_width, fixed_height = reproject_band(
                    rgb[i], profile, target_crs
                )
            else:
                r_band, _, _, _, _ = reproject_band(
                    rgb[i], profile, target_crs,
                    fixed_transform=fixed_transform,
                    fixed_width=fixed_width,
                    fixed_height=fixed_height
                )
            rgb_reproj.append(r_band)
        rgb_reproj = np.stack(rgb_reproj)

        # ✅ RGB 정규화 (0~10000 → 0~255, uint8)
        rgb_scaled = np.clip(rgb_reproj / 10000 * 255, 0, 255).astype(np.uint8)

        # NDVI 재투영 (float32 유지)
        ndvi_reproj, ndvi_profile, _, _, _ = reproject_band(
            ndvi, profile, target_crs,
            fixed_transform=fixed_transform,
            fixed_width=fixed_width,
            fixed_height=fixed_height
        )

        # 저장 경로
        rgb_name = f"{city}_{date_fmt}_RGB.tiff"
        ndvi_name = f"{city}_{date_fmt}_NDVI.tiff"

        rgb_out = os.path.join(rgb_dir, rgb_name)
        ndvi_out = os.path.join(ndvi_dir, ndvi_name)

        # ✅ RGB 저장 (uint8)
        reproj_profile.update({
            "driver": "GTiff",
            "count": 3,
            "dtype": "uint8"
        })
        with rasterio.open(rgb_out, "w", **reproj_profile) as dst:
            for i in range(3):
                dst.write(rgb_scaled[i], i + 1)

        # ✅ NDVI 저장 (float32)
        save_tiff(ndvi_out, ndvi_reproj, ndvi_profile, count=1)

        print(f"[{city}] 저장 완료: {rgb_name}, {ndvi_name}")


In [ ]:
print(f"\n== [{city}] {date_fmt} ==")
print("Red (B04) min/max:", np.min(b04), np.max(b04))
print("NIR (B08) min/max:", np.min(b08), np.max(b08))
print("NDVI min/max:", np.min(ndvi), np.max(ndvi))

print("RGB reproj min/max:")
for i, name in enumerate(["Red", "Green", "Blue"]):
    print(f"  {name}: {np.min(rgb_reproj[i])} ~ {np.max(rgb_reproj[i])}")


In [ ]:
import os

# 테스트할 SAFE 경로 지정 (예시: Riyadh 폴더 중 하나)
safe_path = r"E:\projects\SpaceUp\Track2\sentinel-2\Riyadh\S2B_MSIL2A_20240623T072619_N0510_R049_T38RPN_20240623T085725.SAFE"

def list_r10m_band_files(safe_dir):
    granule_path = os.path.join(safe_dir, "GRANULE")
    if not os.path.exists(granule_path):
        print("❌ GRANULE 폴더 없음")
        return

    l2a_dirs = [d for d in os.listdir(granule_path) if d.startswith("L2A")]
    if not l2a_dirs:
        print("❌ L2A_* 폴더 없음")
        return

    r10m_path = os.path.join(granule_path, l2a_dirs[0], "IMG_DATA", "R10m")
    if not os.path.exists(r10m_path):
        print("❌ R10m 폴더 없음")
        return

    files = os.listdir(r10m_path)
    print(f"\n📂 R10m 폴더에 있는 .jp2 파일 목록:")
    for f in sorted(files):
        print("  ", f)

    # 우리가 원하는 밴드들이 있는지 확인
    for band in ["B02", "B03", "B04", "B08"]:
        found = any(f.endswith(f"{band}_10m.jp2") for f in files)
        print(f"✅ {band} 밴드 찾음" if found else f"❌ {band} 밴드 없음")

list_r10m_band_files(safe_path)


In [ ]:
import rasterio
import numpy as np

# 원하는 밴드 경로 (위에서 확인된 것 중 하나 사용)
band_paths = {
    "B02": r"E:\projects\SpaceUp\Track2\sentinel-2\Riyadh\S2B_MSIL2A_20240623T072619_N0510_R049_T38RPN_20240623T085725.SAFE\GRANULE\L2A_T38RPN_A001234_20240623T072619\IMG_DATA\R10m\T38RPN_20240623T072619_B02_10m.jp2",
    "B03": r"...",
    "B04": r"...",
    "B08": r"..."
}

for band, path in band_paths.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        print(f"\n📊 {band} 통계:")
        print(f"  min: {np.min(arr)}")
        print(f"  max: {np.max(arr)}")
        print(f"  mean: {np.mean(arr)}")


In [ ]:
import os

granule_path = r"E:\projects\SpaceUp\Track2\sentinel-2\Riyadh\S2B_MSIL2A_20240623T072619_N0510_R049_T38RPN_20240623T085725.SAFE\GRANULE"

l2a_dirs = [d for d in os.listdir(granule_path) if d.startswith("L2A")]
l2a_dirs.sort(key=lambda x: len(x))  # 가장 짧은 이름 우선

print("GRANULE 내 L2A 디렉토리 목록:")
for d in l2a_dirs:
    print("  ", d)

print("\n✅ 선택된 디렉토리:", l2a_dirs[0])


In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from datetime import datetime

# 도시 목록 및 기본 설정
base_dir = r"E:\projects\SpaceUp\Track2\sentinel-2"
cities = ["Dammam", "Dhahran", "Madinah", "Riyadh"]
target_crs = "EPSG:4326"

def find_valid_l2a(granule_path):
    """유효한 L2A 디렉토리 자동 선택"""
    if not os.path.exists(granule_path):
        return None
    candidates = [d for d in os.listdir(granule_path) if d.startswith("L2A")]
    for d in candidates:
        r10m_path = os.path.join(granule_path, d, "IMG_DATA", "R10m")
        if not os.path.exists(r10m_path):
            continue
        files = os.listdir(r10m_path)
        if all(any(f.endswith(f"{band}_10m.jp2") for f in files) for band in ["B02", "B03", "B04", "B08"]):
            return os.path.join(granule_path, d, "IMG_DATA", "R10m")
    return None

def load_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
    return arr, profile

def reproject_band(array, profile, dst_crs, fixed_transform=None, fixed_width=None, fixed_height=None):
    src_crs = profile["crs"]
    height, width = array.shape

    if fixed_transform is None:
        with rasterio.io.MemoryFile() as memfile:
            with memfile.open(
                driver="GTiff",
                height=height,
                width=width,
                count=1,
                dtype="float32",
                crs=src_crs,
                transform=profile["transform"]
            ) as tmp_dst:
                tmp_dst.write(array, 1)
                bounds = tmp_dst.bounds

        transform, width, height = calculate_default_transform(
            src_crs, dst_crs, width, height,
            left=bounds.left, bottom=bounds.bottom,
            right=bounds.right, top=bounds.top
        )
    else:
        transform = fixed_transform
        width = fixed_width
        height = fixed_height

    dst_array = np.empty((height, width), dtype="float32")
    reproject(
        source=array,
        destination=dst_array,
        src_transform=profile["transform"],
        src_crs=src_crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

    profile.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "dtype": "float32"
    })

    return dst_array, profile, transform, width, height

def save_tiff(path, array, profile, count=1):
    profile.update(driver="GTiff", count=count)
    with rasterio.open(path, "w", **profile) as dst:
        if count == 1:
            dst.write(array, 1)
        else:
            for i in range(count):
                dst.write(array[i], i + 1)

# === 메인 처리 ===
for city in cities:
    city_path = os.path.join(base_dir, city)
    rgb_dir = os.path.join(city_path, "RGB")
    ndvi_dir = os.path.join(city_path, "NDVI")
    os.makedirs(rgb_dir, exist_ok=True)
    os.makedirs(ndvi_dir, exist_ok=True)

    for item in os.listdir(city_path):
        if not item.endswith(".SAFE"):
            continue

        safe_path = os.path.join(city_path, item)
        granule_path = os.path.join(safe_path, "GRANULE")
        r10m_path = find_valid_l2a(granule_path)

        if not r10m_path:
            print(f"❌ [건너뜀] {city} - {item}에서 유효한 R10m 밴드 경로를 찾을 수 없음")
            continue

        # 날짜 추출
        try:
            date_str = item.split("_")[2][:8]
            date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y%m%d")
        except:
            date_fmt = "unknown"

        # 밴드 경로 지정
        band_paths = {}
        for band in ["B02", "B03", "B04", "B08"]:
            for f in os.listdir(r10m_path):
                if f.endswith(f"{band}_10m.jp2"):
                    band_paths[band] = os.path.join(r10m_path, f)
                    break

        if len(band_paths) < 4:
            print(f"❌ [건너뜀] {city} - {item} 밴드 누락")
            continue

        # 밴드 불러오기
        b02, profile = load_band(band_paths["B02"])
        b03, _ = load_band(band_paths["B03"])
        b04, _ = load_band(band_paths["B04"])
        b08, _ = load_band(band_paths["B08"])

        # NDVI 계산
        ndvi = (b08 - b04) / (b08 + b04 + 1e-6)

        # RGB 구성
        rgb = np.stack([b04, b03, b02])

        # RGB 재투영
        rgb_reproj = []
        fixed_transform = None
        fixed_width = None
        fixed_height = None
        reproj_profile = None

        for i in range(3):
            if i == 0:
                r_band, reproj_profile, fixed_transform, fixed_width, fixed_height = reproject_band(
                    rgb[i], profile, target_crs
                )
            else:
                r_band, _, _, _, _ = reproject_band(
                    rgb[i], profile, target_crs,
                    fixed_transform=fixed_transform,
                    fixed_width=fixed_width,
                    fixed_height=fixed_height
                )
            rgb_reproj.append(r_band)
        rgb_reproj = np.stack(rgb_reproj)

        # ✅ 밴드별 min/max 기반 정규화 (밝기 왜곡 방지)
        rgb_scaled = np.empty_like(rgb_reproj, dtype=np.uint8)
        for i in range(3):
            band = rgb_reproj[i]
            band_min, band_max = np.min(band), np.max(band)
            rgb_scaled[i] = np.clip((band - band_min) / (band_max - band_min) * 255, 0, 255).astype(np.uint8)

        # NDVI 재투영
        ndvi_reproj, ndvi_profile, _, _, _ = reproject_band(
            ndvi, profile, target_crs,
            fixed_transform=fixed_transform,
            fixed_width=fixed_width,
            fixed_height=fixed_height
        )

        # 저장 경로
        rgb_name = f"{city}_{date_fmt}_RGB.tiff"
        ndvi_name = f"{city}_{date_fmt}_NDVI.tiff"

        rgb_out = os.path.join(rgb_dir, rgb_name)
        ndvi_out = os.path.join(ndvi_dir, ndvi_name)

        # RGB 저장 (uint8)
        reproj_profile.update({
            "driver": "GTiff",
            "count": 3,
            "dtype": "uint8"
        })
        with rasterio.open(rgb_out, "w", **reproj_profile) as dst:
            for i in range(3):
                dst.write(rgb_scaled[i], i + 1)

        # NDVI 저장 (float32)
        save_tiff(ndvi_out, ndvi_reproj, ndvi_profile, count=1)

        print(f"✅ 저장 완료: {rgb_name}, {ndvi_name}")


In [ ]:
############## 이 코드 제대로 돌아감 ##############

import os
import numpy as np
import rasterio
from datetime import datetime

# 도시 목록 및 경로 설정
base_dir = r"E:\projects\SpaceUp\Track2\sentinel-2"
cities = ["Dammam", "Dhahran", "Madinah", "Riyadh"]

def find_valid_r10m_path(granule_path):
    """B02, B03, B04, B08이 모두 존재하는 R10m 경로 반환"""
    candidates = [d for d in os.listdir(granule_path) if d.startswith("L2A")]
    for d in candidates:
        r10m_path = os.path.join(granule_path, d, "IMG_DATA", "R10m")
        if not os.path.exists(r10m_path):
            continue
        files = os.listdir(r10m_path)
        if all(any(f.endswith(f"{band}_10m.jp2") for f in files) for band in ["B02", "B03", "B04", "B08"]):
            return r10m_path
    return None

def load_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
    return arr, profile

def save_tiff(path, array, profile, count=1, dtype="float32"):
    profile.update(driver="GTiff", count=count, dtype=dtype)
    with rasterio.open(path, "w", **profile) as dst:
        if count == 1:
            dst.write(array, 1)
        else:
            for i in range(count):
                dst.write(array[i], i + 1)

# === 메인 실행 ===
for city in cities:
    city_path = os.path.join(base_dir, city)
    rgb_dir = os.path.join(city_path, "RGB")
    ndvi_dir = os.path.join(city_path, "NDVI")
    os.makedirs(rgb_dir, exist_ok=True)
    os.makedirs(ndvi_dir, exist_ok=True)

    for item in os.listdir(city_path):
        if not item.endswith(".SAFE"):
            continue

        safe_path = os.path.join(city_path, item)
        granule_path = os.path.join(safe_path, "GRANULE")
        r10m_path = find_valid_r10m_path(granule_path)

        if not r10m_path:
            print(f"❌ [건너뜀] {city} - {item}에서 유효한 R10m 경로를 찾을 수 없음")
            continue

        try:
            date_str = item.split("_")[2][:8]
            date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y%m%d")
        except:
            date_fmt = "unknown"

        # 밴드 경로 추출
        band_paths = {}
        for band in ["B02", "B03", "B04", "B08"]:
            for f in os.listdir(r10m_path):
                if f.endswith(f"{band}_10m.jp2"):
                    band_paths[band] = os.path.join(r10m_path, f)
                    break

        if len(band_paths) < 4:
            print(f"❌ [건너뜀] {city} - {item} 밴드 누락")
            continue

        # 밴드 불러오기
        b02, profile = load_band(band_paths["B02"])
        b03, _ = load_band(band_paths["B03"])
        b04, _ = load_band(band_paths["B04"])
        b08, _ = load_band(band_paths["B08"])

        # NDVI 계산
        ndvi = (b08 - b04) / (b08 + b04 + 1e-6)

        # NDVI 저장
        ndvi_out = os.path.join(ndvi_dir, f"{city}_{date_fmt}_NDVI.tiff")
        save_tiff(ndvi_out, ndvi, profile, count=1, dtype="float32")

        # RGB 생성 (B04, B03, B02)
        rgb = np.stack([b04, b03, b02])
        rgb_scaled = np.empty_like(rgb, dtype=np.uint8)

        # ✅ 1~99% 퍼센타일 정규화
        for i in range(3):
            band = rgb[i]
            p1, p99 = np.percentile(band, 1), np.percentile(band, 99)
            rgb_scaled[i] = np.clip((band - p1) / (p99 - p1) * 255, 0, 255).astype(np.uint8)


        # RGB 저장
        rgb_out = os.path.join(rgb_dir, f"{city}_{date_fmt}_RGB.tiff")
        rgb_profile = profile.copy()
        save_tiff(rgb_out, rgb_scaled, rgb_profile, count=3, dtype="uint8")

        print(f"✅ 저장 완료: {rgb_out}, {ndvi_out}")


In [ ]:
import os
import numpy as np
import rasterio
from datetime import datetime

# 도시 목록 및 경로 설정
base_dir = r"E:\projects\SpaceUp\Track2\sentinel-2"
cities = ["Dammam", "Dhahran", "Madinah", "Riyadh"]

def find_valid_r10m_path(granule_path):
    """B02, B03, B04, B08이 모두 존재하는 R10m 경로 반환"""
    candidates = [d for d in os.listdir(granule_path) if d.startswith("L2A")]
    for d in candidates:
        r10m_path = os.path.join(granule_path, d, "IMG_DATA", "R10m")
        if not os.path.exists(r10m_path):
            continue
        files = os.listdir(r10m_path)
        if all(any(f.endswith(f"{band}_10m.jp2") for f in files) for band in ["B02", "B04", "B08"]):
            return r10m_path
    return None

def load_and_normalize_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
    # 무조건 10000으로 나누고, 0~1 사이로 클리핑
    arr = np.clip(arr / 10000.0, 0, 1)
    return arr, profile

def save_tiff(path, array, profile, count=1, dtype="float32", nodata_val=np.nan):
    if os.path.exists(path):
        os.remove(path)
    profile.update(driver="GTiff", count=count, dtype=dtype, nodata=nodata_val)
    with rasterio.open(path, "w", **profile) as dst:
        if count == 1:
            dst.write(array, 1)
        else:
            for i in range(count):
                dst.write(array[i], i + 1)

# === 메인 실행 ===
for city in cities:
    city_path = os.path.join(base_dir, city)
    evi_dir = os.path.join(city_path, "EVI")
    os.makedirs(evi_dir, exist_ok=True)

    for item in os.listdir(city_path):
        if not item.endswith(".SAFE"):
            continue

        safe_path = os.path.join(city_path, item)
        granule_path = os.path.join(safe_path, "GRANULE")
        r10m_path = find_valid_r10m_path(granule_path)

        if not r10m_path:
            print(f"❌ [건너뜀] {city} - {item}에서 유효한 R10m 경로를 찾을 수 없음")
            continue

        try:
            date_str = item.split("_")[2][:8]
            date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y%m%d")
        except:
            date_fmt = "unknown"

        # 밴드 경로 추출
        band_paths = {}
        for band in ["B02", "B04", "B08"]:
            for f in os.listdir(r10m_path):
                if f.endswith(f"{band}_10m.jp2"):
                    band_paths[band] = os.path.join(r10m_path, f)
                    break

        if len(band_paths) < 3:
            print(f"❌ [건너뜀] {city} - {item} 밴드 누락")
            continue

        # 밴드 불러오기 및 정규화
        b02, profile = load_and_normalize_band(band_paths["B02"])  # Blue
        b04, _ = load_and_normalize_band(band_paths["B04"])        # Red
        b08, _ = load_and_normalize_band(band_paths["B08"])        # NIR

        # EVI 계산
        G, C1, C2, L = 2.5, 6.0, 7.5, 1.0
        valid = (b02 > 0) & (b04 > 0) & (b08 > 0)
        evi = np.full_like(b08, np.nan, dtype="float32")
        evi[valid] = G * (b08[valid] - b04[valid]) / (
            b08[valid] + C1 * b04[valid] - C2 * b02[valid] + L + 1e-6
        )

        # 저장
        evi_out = os.path.join(evi_dir, f"{city}_{date_fmt}_EVI.tiff")
        save_tiff(evi_out, evi, profile, count=1, dtype="float32", nodata_val=np.nan)

        # 디버깅 출력
        print(f"✅ 저장 완료: {evi_out}")
        print("▶ EVI min/max:", np.nanmin(evi), "/", np.nanmax(evi))


In [ ]:
print("🔎 B02 max/min:", b02.max(), b02.min())
print("🔎 B04 max/min:", b04.max(), b04.min())
print("🔎 B08 max/min:", b08.max(), b08.min())


In [ ]:
############## 이 코드 제대로 돌아감 ##############

import os
import numpy as np
import rasterio
from datetime import datetime

# 도시 목록 및 경로 설정
base_dir = r"E:\projects\SpaceUp\Track2\sentinel-2"
cities = ["Dammam", "Dhahran", "Madinah", "Riyadh"]

def find_valid_r10m_path(granule_path):
    """B02, B03, B04, B08이 모두 존재하는 R10m 경로 반환"""
    candidates = [d for d in os.listdir(granule_path) if d.startswith("L2A")]
    for d in candidates:
        r10m_path = os.path.join(granule_path, d, "IMG_DATA", "R10m")
        if not os.path.exists(r10m_path):
            continue
        files = os.listdir(r10m_path)
        if all(any(f.endswith(f"{band}_10m.jp2") for f in files) for band in ["B02", "B04", "B08"]):
            return r10m_path
    return None

def load_and_normalize_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
    arr = np.clip(arr / 10000.0, 0, 1)  # 무조건 정규화 + 클리핑
    return arr, profile

def save_tiff(path, array, profile, count=1, dtype="float32", nodata_val=np.nan):
    if os.path.exists(path):
        os.remove(path)
    profile.update(driver="GTiff", count=count, dtype=dtype, nodata=nodata_val)
    with rasterio.open(path, "w", **profile) as dst:
        if count == 1:
            dst.write(array, 1)
        else:
            for i in range(count):
                dst.write(array[i], i + 1)

# === 메인 실행 ===
for city in cities:
    city_path = os.path.join(base_dir, city)
    evi_dir = os.path.join(city_path, "EVI")
    os.makedirs(evi_dir, exist_ok=True)

    for item in os.listdir(city_path):
        if not item.endswith(".SAFE"):
            continue

        safe_path = os.path.join(city_path, item)
        granule_path = os.path.join(safe_path, "GRANULE")
        r10m_path = find_valid_r10m_path(granule_path)

        if not r10m_path:
            print(f"❌ [건너뜀] {city} - {item}에서 유효한 R10m 경로를 찾을 수 없음")
            continue

        try:
            date_str = item.split("_")[2][:8]
            date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y%m%d")
        except:
            date_fmt = "unknown"

        # 밴드 경로 추출
        band_paths = {}
        for band in ["B02", "B04", "B08"]:
            for f in os.listdir(r10m_path):
                if f.endswith(f"{band}_10m.jp2"):
                    band_paths[band] = os.path.join(r10m_path, f)
                    break

        if len(band_paths) < 3:
            print(f"❌ [건너뜀] {city} - {item} 밴드 누락")
            continue

        # 밴드 불러오기 및 정규화
        b02, profile = load_and_normalize_band(band_paths["B02"])  # Blue
        b04, _ = load_and_normalize_band(band_paths["B04"])        # Red
        b08, _ = load_and_normalize_band(band_paths["B08"])        # NIR

        # EVI 계산
        G, C1, C2, L = 2.5, 6.0, 7.5, 1.0
        valid = (b02 > 0) & (b04 > 0) & (b08 > 0)
        evi = np.full_like(b08, np.nan, dtype="float32")
        evi[valid] = G * (b08[valid] - b04[valid]) / (
            b08[valid] + C1 * b04[valid] - C2 * b02[valid] + L + 1e-6
        )

        # EVI 값 범위 제한 (-1 ~ 1)
        evi = np.clip(evi, -1.0, 1.0)

        # 저장
        evi_out = os.path.join(evi_dir, f"{city}_{date_fmt}_EVI.tiff")
        save_tiff(evi_out, evi, profile, count=1, dtype="float32", nodata_val=np.nan)

        # 디버깅 출력
        print(f"✅ 저장 완료: {evi_out}")
        print("▶ EVI min/max:", np.nanmin(evi), "/", np.nanmax(evi))


In [ ]:
############## 이 코드 제대로 돌아감 ##############

import os
import numpy as np
import rasterio

# 경로 설정
base_dir = r"E:\projects\SpaceUp\Track2\sentinel-2"
cities = ["Dammam", "Dhahran", "Madinah", "Riyadh"]

def save_tiff(path, array, profile, count=1, dtype="float32", nodata_val=np.nan):
    if os.path.exists(path):
        os.remove(path)
    profile.update(driver="GTiff", count=count, dtype=dtype, nodata=nodata_val)
    with rasterio.open(path, "w", **profile) as dst:
        if count == 1:
            dst.write(array, 1)
        else:
            for i in range(count):
                dst.write(array[i], i + 1)

# === LAI 계산 및 저장 ===
for city in cities:
    city_path = os.path.join(base_dir, city)
    evi_dir = os.path.join(city_path, "EVI")
    lai_dir = os.path.join(city_path, "LAI")
    os.makedirs(lai_dir, exist_ok=True)

    for filename in os.listdir(evi_dir):
        if not filename.endswith("_EVI.tiff"):
            continue

        evi_path = os.path.join(evi_dir, filename)
        lai_path = os.path.join(lai_dir, filename.replace("_EVI.tiff", "_LAI.tiff"))

        # 이미 LAI 파일이 있으면 건너뜀
        if os.path.exists(lai_path):
            print(f"⏩ 이미 존재함: {lai_path}")
            continue

        with rasterio.open(evi_path) as src:
            evi = src.read(1).astype("float32")
            profile = src.profile

        # LAI 계산
        lai = 3.618 * evi - 0.118
        lai = np.where(np.isnan(evi), np.nan, lai)
        lai = np.clip(lai, 0, None)  # 음수는 0으로 잘림

        # 저장
        save_tiff(lai_path, lai, profile, count=1, dtype="float32", nodata_val=np.nan)
        print(f"✅ 저장 완료: {lai_path}")
        print("▶ LAI min/max:", np.nanmin(lai), "/", np.nanmax(lai))
